In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://www.jbiet.edu.in/"

html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")

pdf_urls = []

for link in soup.find_all("a", href=True):
    href = link["href"]

    if href.lower().endswith(".pdf"):
        pdf_url = urljoin(url, href)
        pdf_urls.append(pdf_url)

print("Total PDFs:", len(pdf_urls))

In [ ]:
import csv

with open("pdf_urls.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    writer.writerow(["pdf_url"])

    for pdf_url in pdf_urls:
        writer.writerow([pdf_url])

print(f"Saved {len(pdf_urls)} PDF URLs to pdf_urls.csv")

In [ ]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print(pytesseract.get_tesseract_version())

In [ ]:
import csv
import requests
import os

os.makedirs("data/pdfs", exist_ok=True)

with open("pdf_urls.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for i, row in enumerate(reader, start=1):

        pdf_url = row["pdf_url"]

        filename = pdf_url.split("/")[-1]

        filepath = os.path.join("data/pdfs", filename)

        response = requests.get(pdf_url)

        if response.status_code == 200:
            with open(filepath, "wb") as pdf_file:
                pdf_file.write(response.content)

            print(f"{i}. Downloaded: {filename}")

        else:
            print(f"{i}. Failed: {pdf_url}")

In [ ]:
import fitz

pdf_path = "data/pdfs/AICTE-Report-2025-26.pdf"

doc = fitz.open(pdf_path)

for page in doc:
    print(page.get_text())

In [ ]:
import fitz
import os

pdf_folder = "data/pdfs"

pdf_files = [
    file for file in os.listdir(pdf_folder)
    if file.lower().endswith(".pdf")
]

print("Total PDFs:", len(pdf_files))

all_documents = []

for filename in pdf_files:

    pdf_path = os.path.join(pdf_folder, filename)

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    all_documents.append({
        "filename": filename,
        "text": text
    })

    doc.close()

    print("Extracted:", filename)

print("Finished!")

In [ ]:
print(all_documents[5]["filename"])
print(all_documents[5]["text"][:2000])

In [ ]:
for doc in all_documents:
    print(doc["filename"], "→", len(doc["text"]), "characters")

In [ ]:
clean_documents = [
    doc for doc in all_documents
    if doc["text"].strip()
]

print("Total documents:", len(all_documents))
print("Documents with text:", len(clean_documents))

In [ ]:
import re

def clean_text(text):
    # Remove extra spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Remove spaces at the beginning/end
    text = text.strip()

    return text


for doc in clean_documents:
    doc["text"] = clean_text(doc["text"])

print("Text cleaning completed.")

In [ ]:
print(clean_documents[5]["text"][:3000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=200)
chunks=[]

for doc in clean_documents:
    document_chunks=text_splitter.split_text(doc["text"])

    for chunk in document_chunks:
        chunks.append({
            "filename":doc["filename"],
            "text":chunk
        })
print("Total_chunks",len(chunks))

In [ ]:
print(chunks[5])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedded model is loaded")

In [ ]:
test_embedding=embedding_model.embed_query(chunks[0]["text"])

In [ ]:
print(test_embedding[:5])
print("Number of values:", len(test_embedding))

In [ ]:
embeddings = embedding_model.embed_documents(
    [chunk["text"] for chunk in chunks]
)

print("Total embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="jbiET_documents",
    embedding_function=embedding_model,
    persist_directory="data/chroma_db"
)

print("Chroma database created!")

In [ ]:
texts = [chunk["text"] for chunk in chunks]

metadatas = [
    {"filename": chunk["filename"]}
    for chunk in chunks
]

ids = [
    f"chunk_{i}"
    for i in range(len(chunks))
]

vectorstore._collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print("All chunks stored successfully!")

In [ ]:
query = "What courses are offered by JBIET?"

results = vectorstore.similarity_search(query, k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result.metadata["filename"])
    print(result.page_content[:500])

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
print("Retriever created!")

In [ ]:
docs = retriever.invoke("What is the fee for B.Tech at JBIET?")

for doc in docs:
    print(doc.metadata["filename"])
    print(doc.page_content[:300])
    print()

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### LoadingLLM

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print("LLM loaded!")

In [ ]:
llm.invoke("hi")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful chatbot for J.B. Institute of Engineering and Technology (JBIET).

Answer the user's question using ONLY the context provided below.

If the answer cannot be found in the context, say:
"I couldn't find this information in the available JBIET documents."

Do not make up information.

Context:
{context}

Question:
{question}

Answer:
""")

print("Prompt created!")

In [ ]:
def ask_jbiet(question):

    # 1. Retrieve relevant documents
    docs = retriever.invoke(question)

    # 2. Combine the retrieved text
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # 3. Create the prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # 4. Send prompt to LLM
    response = llm.invoke(messages)

    return response.content

In [ ]:
answer = ask_jbiet("what is the fee in jbiet for btech")

print(answer)

In [ ]:
answer = ask_jbiet("What are the admission eligibility requirements at JBIET?")

print(answer)

In [ ]:
answer = ask_jbiet("What facilities are available at JBIET?")

print(answer)

In [ ]:
res=ask_jbiet("What departments are available at JBIET?")
res

In [ ]:
ask_jbiet("What is the admission procedure for B.Tech?")

In [ ]:
ask_jbiet("Who is the current Prime Minister of India?")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

router_prompt = ChatPromptTemplate.from_template("""
You are a question classifier.

Classify the user's question into exactly one of these categories:

JBIET
GENERAL

Choose JBIET if the question is specifically about:
- JBIET college
- J.B. Institute of Engineering and Technology
- JBIET admissions
- JBIET courses
- JBIET fees
- JBIET departments
- JBIET facilities
- JBIET placements
- JBIET rules, regulations, academics, exams, etc.

Choose GENERAL for everything else.

Return ONLY one word:
JBIET
or
GENERAL

Question:
{question}
""")

In [ ]:
def classify_question(question):

    messages = router_prompt.invoke({
        "question": question
    })

    response = llm.invoke(messages)

    if isinstance(response, str):
        return response.strip().upper()

    return response.content.strip().upper()

In [ ]:
print(classify_question("What is the B.Tech fee at JBIET?"))

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template("""
Rewrite the user's latest question as a standalone question.

Use the conversation history to understand references such as:
"it", "they", "that", "what about", "how much", etc.

If the question is already clear and standalone, return it unchanged.

Return ONLY the rewritten question.

Conversation history:
{history}

Latest question:
{question}
""")

In [ ]:
def rewrite_question(question):

    if not chat_history:
        return question

    history = "\n".join(
        f"{message['role']}: {message['content']}"
        for message in chat_history
    )

    messages = rewrite_prompt.invoke({
        "history": history,
        "question": question
    })

    response = llm.invoke(messages)

    return response if isinstance(response, str) else response.content

In [ ]:
chat_history=[]
def chatbot(question):

    # 1. Rewrite the question using conversation history
    standalone_question = rewrite_question(question)

    print("Search question:", standalone_question)

    # 2. Classify the rewritten question
    category = classify_question(standalone_question)

    # 3. JBIET questions → use RAG
    if category == "JBIET":

        docs = retriever.invoke(standalone_question)

        context = "\n\n".join(
            doc.page_content
            for doc in docs
        )

        messages = prompt.invoke({
            "context": context,
            "question": standalone_question
        })

        response = llm.invoke(messages)

    # 4. General questions → use LLM
    else:

        response = llm.invoke(
            standalone_question
        )

    # 5. Get answer
    answer = response if isinstance(response, str) else response.content

    # 6. Save conversation
    chat_history.append({
        "role": "user",
        "content": question
    })

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

In [ ]:
print(chatbot("What is the B.Tech fee at JBIET?"))

In [ ]:
print(chatbot("What about MBA?"))

In [ ]:
print(chatbot("What is machine learning?"))

In [ ]:
import fitz
import pytesseract
from PIL import Image
import io

# Change this to one of your scanned PDF files
pdf_path = "data/pdfs/ServiceRules.pdf"

doc = fitz.open(pdf_path)

# Take the first page
page = doc[0]

# Convert PDF page to an image
pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))

# Convert image bytes to PIL Image
image = Image.open(io.BytesIO(pix.tobytes("png")))

# OCR
text = pytesseract.image_to_string(image)

print(text[:3000])

In [ ]:
import fitz
import pytesseract
from PIL import Image
import io

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages_text = []

    for page in doc:
        # First try normal text extraction
        text = page.get_text().strip()

        # If little/no text was found, use OCR
        if len(text) < 100:
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            image = Image.open(io.BytesIO(pix.tobytes("png")))

            text = pytesseract.image_to_string(image).strip()

        pages_text.append(text)

    doc.close()

    return "\n\n".join(pages_text)

In [ ]:
pdf_path = "data/pdfs/R&D-Cell-Members2026.pdf"

text = extract_pdf_text(pdf_path)

print(text[:5000])

In [ ]:
documents_with_ocr = []

for filename in os.listdir(pdf_folder):

    if filename.lower().endswith(".pdf") and not filename.startswith("~$"):

        pdf_path = os.path.join(pdf_folder, filename)

        print(f"Processing: {filename}")

        text = extract_pdf_text(pdf_path)

        documents_with_ocr.append({
            "filename": filename,
            "text": text
        })

print("\nTotal PDFs processed:", len(documents_with_ocr))

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()

for doc in documents_with_ocr:
    doc["text"] = clean_text(doc["text"])

print("Cleaning completed.")

In [ ]:
for doc in documents_with_ocr:
    if "SERVICE" in doc["text"].upper():
        print(doc["filename"])
        print(doc["text"][:3000])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks_with_ocr = []

for doc in documents_with_ocr:

    document_chunks = text_splitter.split_text(doc["text"])

    for chunk in document_chunks:
        chunks_with_ocr.append({
            "filename": doc["filename"],
            "text": chunk
        })

print("Total chunks:", len(chunks_with_ocr))

In [ ]:
embeddings_ocr = embedding_model.embed_documents(
    [chunk["text"] for chunk in chunks_with_ocr]
)

print("Embeddings created:", len(embeddings_ocr))

In [ ]:
ocr_ids = [f"ocr_chunk_{i}" for i in range(len(chunks_with_ocr))]

vectorstore._collection.add(
    ids=ocr_ids,
    documents=[chunk["text"] for chunk in chunks_with_ocr],
    embeddings=embeddings_ocr,
    metadatas=[
        {"filename": chunk["filename"]}
        for chunk in chunks_with_ocr
    ]
)

print("OCR chunks added to ChromaDB:", len(chunks_with_ocr))

In [ ]:
query = "principal name of jbiet"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("filename"))
    print(doc.page_content[:1500])